# 📗 관측과 궤적 디버깅

55일차에 만든 슈퍼바이저 팀은 최종 답변 하나만 보면 어느 에이전트가 무엇을 받고 무엇을 넘겼는지 알 수 없습니다. 답이 틀렸을 때 원인을 찾으려면 실행 과정 전체를 기록하고 읽어야 합니다. 이 교안에서는 여행자 보험 청구 심사 팀의 실행을 스트리밍과 Langfuse로 기록하고, 그 기록을 거꾸로 따라가 잘못된 실행의 원인을 찾습니다.

<img src="images/00_lesson01_goals.png" width="1050" alt="오늘 배울 것 네 가지: 1 궤적, 곧 에이전트마다 무엇을 받고 무엇을 냈는지 차례로 남긴 기록, 2 스트리밍으로 실행 단계를 번호를 붙여 차례로 출력하기, 3 Langfuse에 세션과 태그를 달아 나무 모양 기록 남기기, 4 잘못된 답변에서 출발해 거꾸로 따라가 원인을 찾고 수정 전과 수정 후 기록을 태그로 나눠 비교하기">

그림의 번호 1~4가 아래 학습 목표 1~4입니다.

**학습 목표**

1. **궤적이 무엇인지 설명하고, 잘못된 결과의 원인이 왜 앞 단계에 있는지 말합니다.** 어떤 노드·에이전트·도구가 어떤 순서로 무엇을 받고 무엇을 냈는지가 궤적입니다.
2. **LangGraph 스트리밍으로 팀의 궤적을 출력합니다.** `subgraphs=True`로 워커 안의 모델·도구 호출까지 번호를 붙여 봅니다.
3. **Langfuse에 세션·태그를 붙여 trace를 남기고 화면을 읽습니다.** trace 트리, Agent Graph, 모델 호출 상세에서 무엇을 보는지 익힙니다.
4. **trace를 거꾸로 따라가 잘못된 실행의 원인을 찾고, 수정 전후 실행을 태그로 비교합니다.** 증상이 난 노드에서 출발해 정보가 사라진 지점까지 거슬러 올라갑니다.

**핵심 개념**

| 개념 | 뜻 | 절 |
|---|---|---|
| 궤적 | 한 번의 실행에서 누가 어떤 순서로 무엇을 받고 무엇을 냈는지 적은 기록 | 1 |
| trace · observation · session | Langfuse의 기록 단위. 실행 한 번, 그 안의 단계 하나, 여러 실행의 묶음 | 5 |
| 태그 | 실행에 붙이는 꼬리표. 수정 전후 실행을 나눠 골라 볼 때 씁니다 | 5, 6 |

## 1. 멀티에이전트는 왜 들여다보기 어려울까

### 1-1. 궤적(trajectory)

**궤적**은 한 번의 실행에서 어떤 노드·에이전트·도구가 어떤 순서로 실행됐고, 각각 무엇을 입력으로 받아 무엇을 냈는지 적은 기록입니다. 에이전트 하나라면 메시지 기록이 곧 궤적이지만, 슈퍼바이저 팀은 바깥 반복(워커 고르기)과 워커 안 반복(도구 고르기)이 겹쳐 있어 기록도 두 겹입니다.

<img src="images/01_trajectory.png" width="1050" alt="보험 청구 CLM-2102 한 건의 실행이 슈퍼바이저, 청구 조회 담당, 슈퍼바이저, 약관 검색 담당, 슈퍼바이저, 답변 작성 담당의 여섯 단계로 이어지고, 단계마다 받은 입력과 낸 출력이 카드로 남는 궤적. 청구 조회 담당과 약관 검색 담당 옆에는 워커 안에서 도구를 부르고 결과를 받는 안쪽 고리가 하나 더 있다">

보험 청구 심사 팀이 CLM-2102를 처리한 궤적의 예:

1. 슈퍼바이저가 심사 요청을 읽고 청구 조회 담당을 고릅니다(지시: CLM-2102 조회).
2. 청구 조회 담당이 청구 조회 도구를 부르고 "5시간 지연, 사유: 출발 3일 전 공지된 노조 파업"을 보고합니다.
3. 슈퍼바이저가 청구 보고를 읽고 약관 검색 담당을 고릅니다.
4. 약관 검색 담당이 약관 검색 도구로 항공기 지연 조항을 찾아 보고합니다.
5. 슈퍼바이저가 두 보고를 읽고 답변 작성 담당을 고릅니다.
6. 답변 작성 담당이 고객 답변을 쓰고, 슈퍼바이저가 답변을 확인한 뒤 끝냅니다.

궤적은 이 여섯 단계 각각에 받은 입력과 낸 출력을 붙인 기록입니다.

### 1-2. 증상과 원인 사이의 거리

**증상**(잘못이 드러난 곳)은 틀린 답이나 이상한 동작이 보이는 노드이고, **원인**(잘못이 생긴 곳)은 그 잘못을 만든 노드나 코드입니다. 멀티에이전트 팀에서는 둘이 다른 경우가 많습니다. 마지막 노드는 앞 노드가 넘겨준 내용대로 답을 썼을 수 있습니다.

<img src="images/02_symptom_to_cause.png" width="1050" alt="식당 메뉴 추천 팀 예시: 고객 요청에는 땅콩 알레르기가 있지만 슈퍼바이저의 지시에서 빠지고(원인), 메뉴 검색 담당은 받은 대로 저녁 인기 메뉴를 검색해 땅콩 소스 비빔국수를 1위로 넘기고, 답변 작성 담당은 받은 대로 써서 땅콩 소스 비빔국수를 추천한다(증상). 아래 주황 화살표를 따라 1 답변, 2 답변 작성 담당, 3 메뉴 검색 담당, 4 슈퍼바이저 순서로 거꾸로 따라간다">

보험 청구 팀과 다른 간단한 예로, 식당의 메뉴 추천 에이전트 팀을 봅시다. 손님이 "땅콩 알레르기가 있어요. 저녁 메뉴를 추천해 주세요."라고 물었는데, 답변은 "땅콩 소스 비빔국수를 추천합니다."였습니다. 답변에서 출발해 앞 단계로 거슬러 올라갑니다.

1. 증상: 답변 작성 담당의 답변이 땅콩이 든 요리를 추천했습니다.
2. 답변 작성 담당의 입력을 열면 받은 추천 후보 1위가 땅콩 소스 비빔국수입니다. 답변 작성 담당은 받은 대로 썼습니다.
3. 메뉴 검색 담당의 도구 호출을 열면 검색어가 "저녁 인기 메뉴"이고 알레르기 조건이 없습니다. 메뉴 검색 담당은 받은 지시대로 검색했습니다.
4. 슈퍼바이저가 메뉴 검색 담당에게 보낸 지시를 열면 "저녁 메뉴를 찾아 주세요"뿐이고 '땅콩 알레르기'가 빠져 있습니다. 고객 요청에 있던 조건이 첫 지시에서 사라졌습니다.

원인은 답변을 쓴 단계보다 두 단계 앞, 슈퍼바이저의 첫 지시에 있습니다. 답변 작성 담당의 프롬프트에 "알레르기를 확인하세요"를 더해도 받은 입력에 알레르기 정보가 없으니 고쳐지지 않습니다. 증상이 난 곳이 아니라 정보가 사라진 곳을 고쳐야 합니다.

여러 에이전트 시스템의 실패 trace 1,600여 건을 분류한 연구(MAST, arXiv 2503.13657)는 실패를 세 범주로 나눕니다. 괄호 안 숫자는 전체 실패에서 차지하는 비율입니다(논문 v3 기준). 대부분이 모델 성능보다 역할·흐름·전달 설계에서 생긴다는 점이 이 교안의 출발점입니다.

| 범주 (비율) | 어디서 생기나 | 대표 실패 (비율) |
|---|---|---|
| 시스템 설계 문제 (43.7%) | 역할 분담·작업 흐름·종료 조건을 잘못 설계함 | 이미 끝낸 단계를 다시 함 (15.7%), 끝낼 시점을 모름 (12.4%), 과제 요구 사항을 어김 (11.8%), 앞선 대화 내용을 잃음 (2.8%) |
| 에이전트 간 어긋남 (32.35%) | 에이전트끼리 정보를 주고받다 어긋남 | 판단한 내용과 실제 행동이 다름 (13.2%), 중요한 정보를 다른 에이전트에 넘기지 않음 (0.85%) |
| 과제 검증 실패 (23.5%) | 결과를 확인하는 단계가 잘못됨 | 확인을 건너뛰거나 대충 함 (8.2%), 엉뚱한 기준으로 확인함 (9.1%) |

어느 범주든 최종 답변만 봐서는 구별되지 않습니다. 궤적을 남겨야 어느 단계에서 생긴 실패인지 가릴 수 있습니다. 6절에서 이런 실패 하나를 실제 trace로 찾습니다.

## 2. 실습 준비

### 2-1. 모델·경로·Langfuse 연결

환경 변수·모델·임베딩 모델 준비(55일차와 같습니다)와 22일차에 쓴 Langfuse 연결을 한 셀에 모았습니다. Langfuse 키가 없으면 아래 순서로 발급합니다.

1. [cloud.langfuse.com](https://cloud.langfuse.com)에 로그인합니다(Google·GitHub 계정 또는 이메일).
2. 처음이면 조직(Organization)과 프로젝트(Project)를 하나 만듭니다.
3. 왼쪽 아래 **Settings**를 누르고 **API Keys**에서 **Create new API keys**를 누릅니다.
4. 화면에 나온 Public Key(`pk-lf-`로 시작), Secret Key(`sk-lf-`로 시작), Host 주소를 단원 폴더 `.env`의 `LANGFUSE_PUBLIC_KEY`, `LANGFUSE_SECRET_KEY`, `LANGFUSE_BASE_URL`에 넣습니다. Secret Key는 이 창을 닫으면 다시 볼 수 없습니다.

아래 셀의 인증 결과가 `False`이면 키와 주소를 고치고 커널을 다시 시작합니다.

In [ ]:
# 실습에 필요한 라이브러리를 불러옵니다.
import os
from pathlib import Path
from dotenv import find_dotenv, load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from IPython.display import Image, Markdown

# material_dir는 output이 있는 단원 폴더입니다.
material_dir = Path(".")
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(parents=True, exist_ok=True)

# 단원 폴더에서 가까운 .env를 찾아 API 키를 읽습니다.
load_dotenv(find_dotenv(usecwd=True))
# 단원 .env에 없는 공통 키는 저장소 최상위 .env에서 읽습니다. 이미 읽은 키는 덮어쓰지 않습니다.
load_dotenv(material_dir.resolve().parent / ".env")

# 답변 모델: .env에 OPENAI_CHAT_MODEL이 없으면 기본 모델을 씁니다.
model_name = os.getenv("OPENAI_CHAT_MODEL", "gpt-5.6-luna")
llm = ChatOpenAI(model=model_name, use_responses_api=True)

# 약관 검색용 임베딩 모델
# check_embedding_ctx_length=False: 입력 길이를 미리 검사해 나누는 자동 처리를 끕니다. 약관 조항은 짧습니다.
embedding_model = OpenAIEmbeddings(model="text-embedding-3-large", dimensions=768,
                                   check_embedding_ctx_length=False)
print("답변 모델:", model_name)


from langfuse import get_client
from langfuse.langchain import CallbackHandler

# Langfuse 클라이언트: .env의 LANGFUSE_PUBLIC_KEY·LANGFUSE_SECRET_KEY·LANGFUSE_BASE_URL을 읽습니다.
langfuse = get_client()

# 키가 틀려도 그래프 실행은 계속되고 기록 전송만 실패하므로, 실행 전에 서버 인증을 확인합니다.
print("Langfuse 인증:", langfuse.auth_check())
print("Langfuse 주소:", os.getenv("LANGFUSE_BASE_URL"))

### 2-2. 보험 청구 자료와 약관 검색

이 교안의 예시는 가상 회사 **한빛손해보험의 여행자 보험 청구 심사 팀**입니다. 청구 2건과 약관 조항 10개를 노트북 안에 둡니다. 지급 여부는 청구 내용만으로 정할 수 없고, 보상 조항과 보상하지 않는 손해 조항, 필요한 서류를 함께 맞춰 봐야 합니다.

자료와 약관 하이브리드 검색기를 한 셀에서 만듭니다. 약관은 "파업", "4시간 이상"처럼 조항에 적힌 낱말이 판정을 가르므로, 53일차의 Chroma 의미 검색기와 BM25 키워드 검색기를 54일차처럼 RRF로 묶습니다. 조항 10개를 임베딩하므로 임베딩 API를 한 번 부릅니다.

In [ ]:
# 실습 기준일(심사일)입니다.
TODAY = "2026-10-08"

# 보험금 청구 데이터: 청구번호별 청구인·연락처·사고 내용·청구액·제출 증빙입니다.
CLAIMS = {
    "CLM-2101": {"청구인": "김하늘", "연락처": "010-2481-3570", "사고 종류": "항공기 지연",
                 "사고일": "2026-09-28", "내용": "인천행 귀국편 6시간 지연", "지연 사유": "태풍으로 인한 기상 악화",
                 "청구 항목": "식비·숙박비", "청구액": 180000, "증빙": "항공사 지연 확인서, 영수증 3장"},
    "CLM-2102": {"청구인": "박도윤", "연락처": "010-7315-2046", "사고 종류": "항공기 지연",
                 "사고일": "2026-09-30", "내용": "다낭행 출국편 5시간 지연", "지연 사유": "출발 3일 전 공지된 항공사 노조 파업",
                 "청구 항목": "식비·숙박비", "청구액": 150000, "증빙": "항공사 지연 확인서, 영수증 2장"},
}

# 여행자 보험 약관 조항 10개: 조항마다 id·제목·본문이 있습니다.
TERMS = [
    {"id": "제1조", "제목": "목적",
     "본문": "이 약관은 한빛손해보험 해외여행자 보험의 보상 범위와 보험금 청구 절차를 정합니다."},
    {"id": "제3조", "제목": "보험기간",
     "본문": "보험기간은 보험증권에 적힌 출국일 0시부터 귀국일 24시까지입니다. 보험기간 밖에서 생긴 손해는 보상하지 않습니다."},
    {"id": "제7조", "제목": "항공기 지연 비용",
     "본문": "탑승 예정 항공기가 4시간 이상 지연되거나 결항되면 그동안 쓴 식비·숙박비·교통비 등 실제 비용을 1회 최대 200,000원까지 보상합니다."},
    {"id": "제8조", "제목": "수하물 지연",
     "본문": "도착한 공항에서 위탁 수하물을 6시간 이상 받지 못하면 그동안 산 생활필수품 비용을 최대 200,000원까지 보상합니다."},
    {"id": "제9조", "제목": "항공기 지연 보상하지 않는 손해",
     "본문": "출발 전에 공지된 파업·쟁의로 생긴 지연, 피보험자의 늦은 탑승 수속 등 고객 사정으로 생긴 지연은 제7조에서 보상하지 않습니다."},
    {"id": "제10조", "제목": "여행 취소",
     "본문": "출국 전 피보험자나 가족의 사망·입원으로 여행을 취소하면 돌려받지 못한 여행 경비를 최대 1,000,000원까지 보상합니다."},
    {"id": "제12조", "제목": "휴대품 손해",
     "본문": "여행 중 휴대품을 도난당하거나 파손되면 1개당 최대 200,000원까지 보상합니다. 도난은 현지 경찰서가 발급한 도난 신고서가 있어야 합니다."},
    {"id": "제13조", "제목": "휴대품 보상하지 않는 손해",
     "본문": "단순 분실, 휴대품을 두고 자리를 떠난 동안의 도난, 현금·유가증권·항공권은 보상하지 않습니다."},
    {"id": "제15조", "제목": "해외 의료비",
     "본문": "여행 중 다치거나 병에 걸려 현지 병원에서 치료받으면 실제 치료비를 최대 3,000,000원까지 보상합니다."},
    {"id": "제18조", "제목": "보험금 청구 서류",
     "본문": "보험금을 청구할 때는 청구서, 사고 증명 서류(항공사 지연 확인서·경찰 도난 신고서·진단서 등), 비용 영수증을 내야 합니다."},
]


from kiwipiepy import Kiwi
from langchain_chroma import Chroma
from langchain_classic.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

# 약관 조항 하나 = 문서 하나. 조항 번호·제목은 메타데이터에도 넣습니다.
term_documents = []
for term in TERMS:
    term_documents.append(Document(page_content=f"{term['제목']}. {term['본문']}",
                                   metadata={"id": term["id"], "제목": term["제목"]}))

# Chroma 벡터 저장소: persist_directory가 없어 메모리에만 둡니다.
vector_store = Chroma(collection_name="day56_travel_terms", embedding_function=embedding_model)

# 조항 id를 문서 id로 써서, 셀을 다시 실행해도 같은 조항이 중복 저장되지 않게 합니다.
term_ids = []
for term in TERMS:
    term_ids.append(term["id"])
vector_store.add_documents(term_documents, ids=term_ids)

# 의미 검색기
dense = vector_store.as_retriever(search_kwargs={"k": 3})

kiwi = Kiwi()


def kiwi_tokenize(text):
    """명사·외국어·숫자만 뽑아 BM25 토큰으로 씁니다."""
    return [token.form.lower() for token in kiwi.tokenize(text)
            if token.tag.startswith("N") or token.tag in {"SL", "SN"}]


# 키워드 검색기(BM25)
bm25 = BM25Retriever.from_documents(term_documents, preprocess_func=kiwi_tokenize, k=3)

# 하이브리드 검색기: BM25와 의미 검색 결과를 RRF로 합칩니다.
terms_retriever = EnsembleRetriever(retrievers=[bm25, dense], weights=[0.5, 0.5], c=60, id_key="id")

## 3. 보험 청구 심사 팀 만들기

55일차 교안 01의 슈퍼바이저 팀 구조를 그대로 씁니다. supervisor가 `Command(goto, update)`로 워커를 고르고, 워커 노드는 안에서 `create_agent` 에이전트를 실행해 보고를 `results`에 더한 뒤 돌아옵니다. 도메인과 워커, 그리고 보고를 프롬프트에 넣는 방식(3-2)이 바뀝니다.

<img src="images/node_flow/l1_team_overview.png" width="1050" alt="supervisor가 claim_worker·terms_worker·reply_writer 중 하나에 일을 맡기고, 워커는 supervisor로 돌아오며, FINISH이면 END로 끝나는 보험 청구 심사 팀 그래프">

1절의 역할 이름이 코드에서는 아래 노드 이름이 됩니다. 슈퍼바이저 노드는 `supervisor`입니다.

| 워커 | 1절의 역할 | 맡은 일 | 도구 |
|---|---|---|---|
| claim_worker | 청구 조회 담당 | 청구 내용 조회 | `get_claim` |
| terms_worker | 약관 검색 담당 | 약관 조항 검색(멀티에이전트 RAG의 검색 담당) | `search_terms` |
| reply_writer | 답변 작성 담당 | 고객 답변 작성 | `get_reply_template` |

### 3-1. 도구와 워커 에이전트

워커마다 그 일에 필요한 도구만 주고 `create_agent`로 워커 에이전트 셋을 만듭니다(55일차와 같아 한 셀에 모았습니다). 이번에는 `name` 인자로 에이전트에 이름을 붙여, 스트리밍과 Langfuse 기록에서 어느 에이전트의 실행인지 바로 구별되게 합니다.

In [ ]:
# 도구(@tool)와 워커 에이전트(create_agent)
from langchain_core.tools import tool
from langchain.agents import create_agent

# 청구 조회 도구: 청구 데이터를 조회만 합니다.
@tool
def get_claim(claim_id: str) -> dict:
    """청구번호로 청구인·사고 종류·사고 내용·지연 사유·청구액·제출 증빙을 조회합니다."""
    # 없는 청구번호면 안내 문장을 돌려줍니다.
    if claim_id not in CLAIMS:
        return {"안내": "없는 청구번호입니다."}

    return CLAIMS[claim_id]


# 답변 양식 도구: 양식만 돌려줍니다. 청구·약관 사실은 전달받은 보고에서 가져옵니다.
@tool
def get_reply_template() -> str:
    """고객에게 보낼 심사 결과 안내의 양식을 반환합니다."""
    return f"""1. 청구 확인 인사 한 줄 (심사일 {TODAY})
2. 확인한 청구 내용(청구번호·사고 종류·청구액)
3. 적용한 약관 조항과 심사 결과(지급·부지급·서류 보완, 지급액)
4. 고객이 할 일 또는 추가 안내"""


# 약관 검색 도구: 하이브리드 검색기로 찾은 조항 중 앞 3개를 돌려줍니다.
@tool
def search_terms(query: str) -> list[dict]:
    """여행자 보험 약관에서 query와 가까운 조항 3개를 찾아 조항 번호·제목·내용을 돌려줍니다."""
    # 두 검색기가 3개씩 찾으므로 합친 결과는 최대 6개입니다. 순위가 높은 3개만 씁니다.
    found = terms_retriever.invoke(query)[:3]

    # 문서 객체 대신 모델이 읽기 쉬운 dict 목록으로 돌려줍니다.
    terms = []
    for doc in found:
        terms.append({"id": doc.metadata["id"], "제목": doc.metadata["제목"], "내용": doc.page_content})

    return terms


# name: 에이전트 그래프의 이름입니다. Langfuse는 이름에 agent가 들어간 실행을 agent 종류로 표시합니다.
# 청구 조회 워커: 조회한 항목을 빠짐없이 보고합니다.
claim_agent = create_agent(llm, tools=[get_claim], name="claim_agent", system_prompt=(
    "한빛손해보험 여행자 보험 청구 심사 팀의 청구 조회 담당입니다. 지시받은 청구를 도구로 조회하고, "
    "도구가 돌려준 항목(사고 종류·내용·지연 사유·청구액·증빙)을 빠짐없이 짧게 보고하세요."))

# 약관 검토 워커: 보상하는 조항과 보상하지 않는 조항을 모두 찾아 두 줄 형식으로 보고합니다.
terms_agent = create_agent(llm, tools=[search_terms], name="terms_agent", system_prompt=(
    "한빛손해보험 여행자 보험 청구 심사 팀의 약관 담당입니다. search_terms로 청구에 관련된 "
    "보상 조항과 보상하지 않는 손해 조항을 모두 찾으세요. 보고는 아래 두 줄 형식으로 씁니다.\n"
    "적용 조항: 보상하는 조항의 번호·제목과 그 조항 본문 문장을 그대로 옮겨 적습니다.\n"
    "보상하지 않는 경우: 이 청구에 해당할 수 있는 보상 제외 조항의 번호·제목과 본문 문장을 그대로 옮겨 적습니다. "
    "해당하는 조항이 없으면 '해당 없음'이라고 씁니다."))

# 답변 작성 워커: 보고에 있는 사실만으로 답변을 씁니다.
reply_agent = create_agent(llm, tools=[get_reply_template], name="reply_agent", system_prompt=(
    "한빛손해보험 여행자 보험 청구 심사 팀의 답변 작성 담당입니다. 양식을 도구로 확인하고, "
    "전달받은 보고에 있는 사실만으로 고객에게 보낼 한국어 답변을 쓰세요. 보고에 없는 약관 내용을 지어내지 마세요."))

### 3-2. 팀 State와 슈퍼바이저 (`supervisor`)

State와 supervisor 노드의 구조는 55일차와 같습니다. 처리 원칙에는 워커를 부르는 순서를 적지 않고 기준만 적습니다. 보고를 프롬프트에 넣는 보조 함수 `format_results`만 55일차와 다릅니다. State와 슈퍼바이저 프롬프트·출력 형식은 한 셀에 모았습니다.

<img src="images/node_flow/l1_team_supervisor.png" width="1050" alt="보험 청구 심사 팀 그래프에서 지금 만드는 노드: supervisor">

In [ ]:
# Literal: Command 반환 타입 힌트에 이동할 노드 이름을 적을 때 씁니다.
from typing import Literal
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command

# [State] 심사 팀 그래프의 공유 데이터
class TeamState(TypedDict):
    request: str  # 심사 요청 문장
    max_turns: int  # 처리 상한: 워커에게 일을 맡길 수 있는 횟수
    instruction: str  # supervisor가 이번 워커에게 준 지시
    visits: list[str]  # 일을 맡긴 워커 순서
    turn: int  # 일을 맡긴 횟수
    results: list[dict]  # [{"worker": 워커 이름, "result": 보고}, ...]


# 워커 소개: 워커마다 맡은 일입니다.
WORKER_GUIDE = (
    "claim_worker: 청구 내용(사고 종류·사유·청구액·증빙) 조회. "
    "terms_worker: 여행자 보험 약관 조항 검색과 보상 여부 근거 정리. "
    "reply_writer: 확인된 내용으로 고객 답변 작성."
)

# 팀의 처리 원칙: 워커를 부르는 순서는 정하지 않고 지켜야 할 기준만 적습니다.
TEAM_RULES = (
    "처리 원칙: "
    "1) 지급·부지급은 청구 내용과, 그 청구에 적용되는 약관 조항(보상하지 않는 경우 포함)이 "
    "보고로 모두 확인된 뒤에만 reply_writer에게 답변을 맡깁니다. "
    "2) 판단에 필요한 사실이 아직 보고에 없으면 그 사실을 알아낼 수 있는 워커에게 맡깁니다. 이미 보고된 내용은 다시 시키지 않습니다. "
    "3) 약관이 요구하는 서류가 빠졌으면 지급을 약속하지 말고 reply_writer에게 서류 보완 안내를 맡깁니다. "
    "4) 고객에게 보낼 답변이 나오면 FINISH를 고릅니다."
)

# 슈퍼바이저 판단의 출력 형식: 다음 워커(next)와 지시(instruction)
class Route(BaseModel):
    next: Literal["claim_worker", "terms_worker", "reply_writer", "FINISH"] = Field(
        description="다음에 일을 맡길 워커. reply_writer가 답변을 썼으면 FINISH")
    instruction: str = Field(
        description="다음 워커에게 줄 지시. 워커가 알아야 할 앞 보고(청구번호·사고 내용·조항)를 넣는다. FINISH이면 빈 문자열")

# 시스템 메시지에 워커 소개와 처리 원칙, 사람 메시지에 심사 요청과 보고를 넣습니다.
supervisor_prompt = ChatPromptTemplate.from_messages([
    ("system", "한빛손해보험 여행자 보험 청구 심사 팀의 팀장입니다. 직접 조회하거나 판단 근거를 지어내지 않고 워커에게 맡깁니다. "
     + WORKER_GUIDE + " " + TEAM_RULES +
     " 심사 요청과 지금까지 보고를 읽고 다음 워커 한 명과 지시를 정하세요."),
    ("human", "심사 요청: {request}\n\n지금까지 보고:\n{results}"),
])

# 프롬프트 결과를 LLM에 넘겨 Route 형식으로 받습니다.
supervisor_chain = supervisor_prompt | llm.with_structured_output(Route, strict=True)

`format_results`는 보고 목록을 '[워커] 보고' 문단으로 바꿔 supervisor 프롬프트와 워커 메시지에 넣는 보조 함수입니다. 노드 앞부분의 `if` 문은 55일차의 처리 상한 검사입니다.

In [ ]:
# [보조 함수] LLM 컨텍스트 만들기: 워커 보고 목록을 supervisor 프롬프트와 워커 메시지에 넣을 '[워커] 보고' 문단으로 바꿉니다.
def format_results(results):
    """워커 보고 목록을 '[워커] 보고' 문단으로 묶습니다."""
    # 아직 보고가 없으면 첫 판단이라는 뜻으로 '아직 없음'을 넘깁니다.
    if not results:
        return "아직 없음"

    # 보고가 쌓일수록 프롬프트가 길어지므로, 토큰을 아끼려고 보고마다 앞 60자만 넣습니다.
    lines = []
    for row in results:
        summary = row["result"][:60]
        lines.append(f"[{row['worker']}] {summary}")

    return "\n\n".join(lines)


# [노드] 슈퍼바이저: 보고를 읽고 다음 워커와 지시를 정합니다.
def supervisor(state: TeamState) -> Command[Literal["claim_worker", "terms_worker", "reply_writer", END]]:
    """지금까지 보고를 읽고 다음 워커와 지시를 정해 Command로 보냅니다."""
    # 처리 상한에 닿으면 더 맡기지 않고 심사 담당자에게 넘기는 답변으로 마무리합니다.
    if state["turn"] >= state["max_turns"]:
        # reply_writer가 이미 답변을 썼으면 바로 끝냅니다.
        if "reply_writer" in state["visits"]:
            return Command(goto=END)

        # 아직 답변이 없으면 reply_writer에게 심사 담당자 확인 안내를 한 번 맡깁니다.
        return Command(goto="reply_writer", update={
            "instruction": ("처리 상한에 닿았습니다. 지급을 약속하지 말고, 지금까지 확인한 내용과 함께 "
                            "'심사 담당자 확인 필요'로 안내하세요."),
            "visits": state["visits"] + ["reply_writer"],
            "turn": state["turn"] + 1,
        })

    # 심사 요청과 지금까지의 보고를 넘겨, 다음 워커와 지시를 Route 형식으로 받습니다.
    route = supervisor_chain.invoke({
        "request": state["request"],
        "results": format_results(state["results"]),
    })

    # FINISH를 고르면 State를 바꾸지 않고 끝냅니다.
    if route.next == "FINISH":
        return Command(goto=END)

    # goto: 고른 워커 / update: 그 워커가 읽을 지시와 방문 기록·턴 수
    return Command(goto=route.next, update={
        "instruction": route.instruction,
        "visits": state["visits"] + [route.next],
        "turn": state["turn"] + 1,
    })

### 3-3. 워커 노드와 그래프 조립

`run_worker`는 심사 요청, 지시, 지금까지 받은 보고를 워커 에이전트에 전달하고 새 보고를 덧붙인 `results`를 반환합니다. 세 워커 노드가 함께 씁니다. 워커 노드 셋과 그래프 조립은 55일차와 같은 모양이라 한 셀에 모았습니다. 엣지는 `START -> supervisor` 하나만 등록하고, 이동은 노드가 반환하는 `Command`가 정합니다.

<img src="images/node_flow/l1_team_workers.png" width="1050" alt="보험 청구 심사 팀 그래프에서 지금 만드는 노드: claim_worker, terms_worker, reply_writer">

그림에서 주황색으로 표시한 워커 노드 셋(`claim_worker`, `terms_worker`, `reply_writer`)을 아래 셀 하나에서 만들고 팀 그래프로 조립합니다.

In [ ]:
# [보조 함수] 워커 실행: 심사 요청·지시·지금까지의 보고로 워커 에이전트의 메시지를 만들고, 새 보고를 덧붙인 목록을 반환합니다.
def run_worker(agent, worker, state):
    """심사 요청·지시·지금까지 보고를 워커 에이전트에 전달하고, 새 보고를 덧붙인 results를 반환합니다."""
    # 워커끼리는 직접 대화하지 않으므로 팀이 지금까지 확인한 내용을 메시지에 넣어 줍니다.
    message = (f"심사 요청: {state['request']}\n\n지시: {state['instruction']}\n\n"
               f"팀이 지금까지 확인한 내용:\n{format_results(state['results'])}")

    reply = agent.invoke({"messages": [{"role": "user", "content": message}]})

    # 마지막 메시지(워커의 최종 보고)만 꺼내 기존 보고 뒤에 붙인 새 목록을 반환합니다.
    return state["results"] + [{"worker": worker, "result": reply["messages"][-1].text}]


# [노드] 청구 조회 워커
def claim_worker(state: TeamState) -> Command[Literal["supervisor"]]:
    """claim_agent의 보고를 results에 더하고 supervisor로 돌아갑니다."""
    # 워커 에이전트를 먼저 실행하고, 새 보고가 더해진 results를 update에 담습니다.
    results = run_worker(claim_agent, "claim_worker", state)
    return Command(goto="supervisor", update={"results": results})


# [노드] 약관 검토 워커
def terms_worker(state: TeamState) -> Command[Literal["supervisor"]]:
    """terms_agent의 보고를 results에 더하고 supervisor로 돌아갑니다."""
    # 워커 에이전트를 먼저 실행하고, 새 보고가 더해진 results를 update에 담습니다.
    results = run_worker(terms_agent, "terms_worker", state)
    return Command(goto="supervisor", update={"results": results})


# [노드] 답변 작성 워커
def reply_writer(state: TeamState) -> Command[Literal["supervisor"]]:
    """reply_agent의 보고를 results에 더하고 supervisor로 돌아갑니다."""
    # 워커 에이전트를 먼저 실행하고, 새 보고가 더해진 results를 update에 담습니다.
    results = run_worker(reply_agent, "reply_writer", state)
    return Command(goto="supervisor", update={"results": results})


# [그래프 조립] 슈퍼바이저와 워커 셋을 노드로 등록합니다.
team_builder = StateGraph(TeamState)
team_builder.add_node("supervisor", supervisor)
team_builder.add_node("claim_worker", claim_worker)
team_builder.add_node("terms_worker", terms_worker)
team_builder.add_node("reply_writer", reply_writer)

# 시작 엣지만 등록합니다. 나머지 이동은 노드가 반환하는 Command가 정합니다.
team_builder.add_edge(START, "supervisor")

team_app = team_builder.compile()

# 컴파일한 노드와 연결을 이미지로 확인합니다. 점선은 반환 타입 힌트 Command[Literal[...]]로 그립니다.
display(Image(team_app.get_graph().draw_mermaid_png()))

이 교안은 같은 팀을 청구마다 여러 번 실행합니다. 첫 입력을 만드는 함수를 하나 둡니다.

In [ ]:
# 처리 상한: 워커에게 일을 맡길 수 있는 횟수입니다.
MAX_TURNS = 8


# [보조 함수] 첫 입력 만들기: 같은 팀을 여러 번 실행하므로 첫 입력의 키와 초깃값을 이 함수에서만 정합니다.
def claim_input(claim_id):
    """청구번호로 팀 그래프의 첫 입력을 만듭니다."""
    # 보고·방문 기록은 비우고 턴은 0부터 셉니다.
    return {
        "request": f"{claim_id} 보험금 청구를 심사하고 고객 답변을 써 주세요.",
        "max_turns": MAX_TURNS,
        "instruction": "",
        "visits": [],
        "turn": 0,
        "results": [],
    }

## 4. 스트리밍으로 궤적 출력하기

Langfuse 없이 LangGraph만으로도 궤적을 볼 수 있습니다. 51일차에 쓴 `stream(stream_mode="updates")`는 노드가 끝날 때마다 그 노드가 State에 쓴 값을 하나씩 내줍니다. 여기에 `subgraphs=True`를 더하면 워커 노드 안에서 실행된 에이전트 그래프(model·tools 노드)의 이벤트까지 받습니다.

`subgraphs=True`이면 이벤트가 `(네임스페이스, 이벤트)` 튜플로 옵니다. **네임스페이스**는 이벤트가 어느 그래프 안에서 났는지 적은 튜플입니다. 팀 그래프에서 난 이벤트는 빈 튜플 `()`이고, 워커 노드 안 에이전트에서 난 이벤트는 `('노드 이름:작업 id',)`입니다. **작업 id**는 LangGraph가 노드 실행 한 번마다 붙이는 고유 번호입니다.

| 받는 값 | 예 | 뜻 |
|---|---|---|
| 네임스페이스 | `()` | 팀 그래프에서 난 이벤트 |
| 네임스페이스 | `('terms_worker:<작업 id>',)` | terms_worker 노드 안에서 실행된 terms_agent의 이벤트 |
| 이벤트 | `{'supervisor': {'instruction': ..., 'visits': ..., 'turn': ...}}` | 노드 이름과 그 노드가 State에 쓴 값 |

워커 노드의 이벤트는 그 안의 에이전트가 모두 끝난 뒤에 옵니다. 그래서 안쪽 model·tools 이벤트가 먼저 출력됩니다.

먼저 이벤트 하나에서 위치와 핵심 값을 꺼내는 보조 함수 둘을 만듭니다.

In [ ]:
# 이벤트의 메시지를 도구 결과와 모델 답으로 가르는 데 씁니다.
from langchain_core.messages import AIMessage, ToolMessage


# [보조 함수] 이벤트 위치: 네임스페이스 튜플을 '팀' 또는 '팀/워커 노드' 문자열로 바꿉니다.
def event_place(namespace):
    """빈 네임스페이스는 '팀', 워커 노드 안의 이벤트는 '팀/워커 노드'로 바꿉니다."""
    names = ["팀"]

    # 항목은 'terms_worker:<작업 id>' 모양이라 ':' 앞의 노드 이름만 씁니다.
    for part in namespace:
        names.append(part.split(":")[0])

    return "/".join(names)


# [보조 함수] 핵심 값 꺼내기: 노드가 State에 쓴 값(update)을 출력할 줄 목록으로 바꿉니다.
def update_lines(update):
    """노드가 반환한 update에서 지시·보고·도구 호출·도구 결과·에이전트 답을 한 줄씩 꺼냅니다."""
    # Command(goto=END)처럼 State를 바꾸지 않은 노드는 update가 None입니다.
    if update is None:
        return ["State 변경 없음"]

    # 팀 그래프의 노드: supervisor는 지시를, 워커 노드는 보고를 씁니다.
    lines = []
    if "instruction" in update:
        lines.append(f"지시({update['visits'][-1]}): {update['instruction']}")
    if "results" in update:
        last = update["results"][-1]
        lines.append(f"보고({last['worker']}): {last['result']}")

    # 워커 에이전트 안의 model·tools 노드는 messages에 새 메시지를 씁니다.
    if "messages" in update:
        for message in update["messages"]:
            if isinstance(message, ToolMessage):
                lines.append(f"도구 결과({message.name}): {message.content}")
            elif isinstance(message, AIMessage) and message.tool_calls:
                for call in message.tool_calls:
                    lines.append(f"도구 호출: {call['name']} {call['args']}")
            elif isinstance(message, AIMessage):
                lines.append(f"에이전트 답: {message.text}")

    return lines

`print_trajectory`는 이벤트마다 번호를 붙여 위치·노드를 한 줄에, 핵심 값을 그 아래 줄에 출력합니다.

In [ ]:
# [보조 함수] 궤적 출력: 이벤트마다 번호·위치·노드를 한 줄, 핵심 값을 그 아래 줄에 출력합니다.
def print_trajectory(app, inputs):
    """그래프를 스트리밍하며 이벤트에 번호를 붙여 차례로 출력합니다."""
    step = 0

    # subgraphs=True: 워커 노드 안에서 실행된 에이전트 그래프의 이벤트도 (네임스페이스, 이벤트)로 받습니다.
    for namespace, event in app.stream(inputs, stream_mode="updates", subgraphs=True):
        place = event_place(namespace)

        # updates 모드의 이벤트는 {노드 이름: 그 노드가 State에 쓴 값}입니다.
        for node, update in event.items():
            step += 1
            print(f"[{step}] {place} > {node}")

            # 한눈에 읽으려고 줄바꿈을 ' / '로 바꾸고 줄마다 120자까지만 출력합니다. 전체 값은 Langfuse에서 봅니다.
            for line in update_lines(update):
                one_line = line.replace("\n", " / ")
                print("    ", one_line[:120])

**실행 확인:** CLM-2101을 실행합니다. 번호마다 `팀 > 노드` 또는 `팀/워커 노드 > model·tools`가 찍히고, 그 아래에 지시·도구 호출·도구 결과·보고가 보여야 합니다.

In [ ]:
# CLM-2101을 Langfuse 없이 스트리밍으로 실행하며 궤적을 출력합니다.
print_trajectory(team_app, claim_input("CLM-2101"))

같은 워커(예: claim_worker)가 여러 번 불렸다면, 그 이유는 6절에서 찾습니다.

스트리밍은 지금 실행을 바로 보기에 좋지만, 노트북을 닫으면 사라지고 모델 호출의 토큰·비용·지연 시간도 나오지 않습니다. 운영에서는 실행 기록을 서버에 남기는 관측 도구를 씁니다.

## 5. Langfuse로 trace 남기기

### 5-1. trace·observation·session

22일차에 Langfuse로 모델 호출 하나를 기록했습니다. 멀티에이전트 팀을 기록하면 한 번의 실행 안에 그래프·노드·에이전트·모델·도구 호출이 겹겹이 들어간 트리가 됩니다.

<img src="images/03_trace_tree.png" width="1050" alt="CLM-2101 심사 한 번의 trace가 claim-team 아래 supervisor, claim_worker, 다시 supervisor, terms_worker 같은 노드 줄로 이어지고, 워커 노드 아래 에이전트, 에이전트 아래 모델 호출(ChatOpenAI)과 도구 호출(get_claim, search_terms), search_terms 아래 검색기 호출(EnsembleRetriever)이 겹쳐진 트리. 줄 하나하나가 observation이고, 같은 청구 한 건(작업 하나)의 첫 실행 trace와, 그 실행이 오류로 멈춘 뒤 멈춘 단계부터 다시 실행한 trace가 한 session으로 묶인다">

| 단위 | 뜻 | 보험 심사 팀에서 |
|---|---|---|
| **trace** | 요청 한 번의 실행 전체 | CLM-2101 심사 한 번 (`team_app.invoke` 1회) |
| **observation** | trace 안의 단계 하나. 종류가 붙습니다 | supervisor 노드 실행, `search_terms` 호출 하나 |
| **session** | 같은 작업의 trace를 묶은 것 | CLM-2101 한 건의 첫 실행과, 그 실행이 오류로 멈춘 뒤 멈춘 단계부터 다시 실행한 기록 |

observation은 종류별로 표시됩니다.

| 종류 | 무엇이 이 종류로 기록되나 | 보험 심사 팀의 예 |
|---|---|---|
| agent | 모델이 도구를 고르고 그 결과를 본 뒤 다시 고르기를 끝날 때까지 되풀이하는 실행(에이전트 루프) | `claim_agent`, `terms_agent`, `reply_agent` (`create_agent`로 만든 워커 에이전트) |
| chain | 코드가 정해 둔 단계를 정해 둔 순서대로 한 번 지나가는 실행 | `claim-team`(팀 그래프), `supervisor`, `claim_worker` |
| generation | 모델 호출(입력·출력·토큰·비용·지연 시간) | supervisor의 Route 판단, 워커 에이전트 안의 model |
| tool | 도구 호출 | `get_claim`, `search_terms` |
| retriever | 검색기 호출 | `search_terms` 안의 `terms_retriever`(화면에는 클래스 이름 `EnsembleRetriever`로 표시) |

**구분:** generation은 모델을 한 번 부른 기록, agent는 모델이 도구를 고르며 여러 번 되풀이한 실행 묶음, chain은 코드가 정한 단계를 순서대로 한 번 지나간 실행 묶음입니다.

### 5-2. 실행 설정 만들기

LangGraph 실행 설정(config)에 Langfuse 콜백과 세션·태그를 함께 담습니다. Langfuse 핸들러는 생성자로 세션·태그를 받지 않고, config의 `metadata`에서 `langfuse_`로 시작하는 키를 읽어 trace에 붙입니다.

| 키 | 값 | 쓰임 |
|---|---|---|
| `callbacks` | `[CallbackHandler()]` | 실행 기록을 Langfuse로 보냅니다 |
| `run_name` | `"claim-team"` | trace 이름 |
| `recursion_limit` | `30` | 그래프가 실행할 단계 수의 상한 |
| `configurable.thread_id` | `claim-review-심사일-청구번호` | 한 작업(청구 한 건)의 id. 체크포인터를 쓰면 오류로 멈춘 실행을 같은 thread_id로 멈춘 단계부터 다시 실행합니다. 이 교안에는 체크포인터가 없어 실행 구분용 이름만 남깁니다 |
| `metadata.langfuse_session_id` | thread_id와 같은 값 | 같은 값의 trace를 Langfuse 세션 하나로 묶습니다 |
| `metadata.langfuse_tags` | 태그 목록 | 수정 전후 실행(v1, v2)을 구분하고, 여러 청구를 함께 걸러 봅니다 |

**thread_id와 세션 id**

둘은 이름이 비슷하지만 읽는 곳과 하는 일이 다릅니다.

| | `thread_id` (LangGraph) | 세션 id (Langfuse) |
|---|---|---|
| 넣는 곳 | `config["configurable"]["thread_id"]` | `config["metadata"]["langfuse_session_id"]` |
| 하는 일 | 체크포인터가 State를 저장하고, 이어 하기 때 멈춘 곳을 찾는 열쇠 | 여러 trace를 화면에서 한 묶음으로 보여 주는 이름 |
| 없으면 | 체크포인터를 쓰는 그래프는 실행되지 않습니다 | trace는 남지만 Session ID 칸이 비어 있습니다. `thread_id`만 넣어서는 세션이 생기지 않습니다 |

이 단원은 **작업 하나 = thread 하나 = 세션 하나**로 두고, 두 값에 같은 이름을 넣습니다. 그러면 한 작업에서 생긴 실행(첫 실행, 오류로 멈춘 뒤 멈춘 단계부터 다시 실행한 것, 사람 확인을 받은 뒤 다시 실행한 것)이 Langfuse 세션 하나에 모입니다. 챗봇에서 대화 하나를 세션 하나로 보는 것과 같은 방식이고, 가장 흔한 기본값입니다.

- **지킬 규칙:** 같은 `thread_id`의 실행은 늘 같은 세션 id로 보냅니다. 첫 실행과, 멈춘 단계부터 다시 실행한 실행의 세션 값이 다르면, 체크포인터로는 이어졌는데 Langfuse에서는 두 세션으로 갈라져 보입니다.
- **다르게 두는 경우:** 하루 처리분처럼 여러 작업을 한 화면에 묶고 싶으면 세션을 작업 묶음 이름으로, `thread_id`를 작업마다 따로 둡니다(세션 하나에 thread 여러 개). 이때도 thread 하나가 두 세션에 걸치면 안 됩니다.
- **여러 작업을 함께 보기:** 이 교안은 세션을 청구마다 따로 두므로, 청구 2건은 태그(`day56`, `v1`)로 걸러 함께 봅니다.

In [ ]:
# [보조 함수] 실행 설정 만들기: 실행마다 새 Langfuse 핸들러와 세션·thread_id·태그를 담습니다.
def make_config(session_id, thread_id, tags):
    """Langfuse 콜백·세션·thread_id·태그를 담은 그래프 실행 설정을 만듭니다."""
    return {
        # 실행마다 새 핸들러를 넣어, 실행이 끝난 뒤 그 실행의 trace id를 핸들러에서 읽습니다.
        "callbacks": [CallbackHandler()],
        # trace 이름: Langfuse 목록에서 팀 그래프 실행을 이 이름으로 찾습니다.
        "run_name": "claim-team",
        # 그래프가 실행할 수 있는 단계 수의 상한입니다.
        "recursion_limit": 30,
        # thread_id는 작업(청구) 하나의 id입니다. 체크포인터가 State를 저장하고 다시 찾는 열쇠입니다.
        "configurable": {"thread_id": thread_id},
        # langfuse_로 시작하는 키는 trace의 세션·태그로 들어갑니다. 세션은 이 키에서만 정해집니다.
        "metadata": {"langfuse_session_id": session_id, "langfuse_tags": tags},
    }

### 5-3. 청구 2건을 태그를 달아 실행하기

청구마다 `make_config`로 새 설정을 만들어 실행합니다. thread_id와 세션 id는 청구마다 `claim-review-심사일-청구번호`이고, 태그는 두 건 모두 `day56`, `v1`입니다. 실행이 끝나면 핸들러의 `last_trace_id`로 그 실행의 trace 주소를 얻습니다(`get_trace_url`). 기록은 모아서 보내므로 끝에 `flush()`를 부릅니다.

| 청구 | 자료 | 기대하는 심사 |
|---|---|---|
| CLM-2101 항공기 지연 | 6시간, 태풍, 180,000원 | 제7조에 따라 180,000원 지급 (1회 한도 200,000원 이내) |
| CLM-2102 항공기 지연 | 5시간, 출발 3일 전 공지된 파업, 150,000원 | 제9조에 따라 보상하지 않음 |

In [ ]:
# 청구 2건을 청구마다 자기 thread·세션으로, 같은 태그를 달아 실행하고 방문 순서·보고·trace 주소를 모읍니다.
claim_runs = {}
for claim_id in CLAIMS:
    run_id = f"claim-review-{TODAY}-{claim_id}"
    config = make_config(run_id, run_id, ["day56", "v1"])
    result = team_app.invoke(claim_input(claim_id), config)

    # 실행이 끝나면 핸들러가 이번 실행의 trace id를 기억합니다.
    handler = config["callbacks"][0]
    claim_runs[claim_id] = {
        "visits": result["visits"],
        "turn": result["turn"],
        "results": result["results"],
        "trace_url": langfuse.get_trace_url(trace_id=handler.last_trace_id),
    }
    print(claim_id, "방문 순서:", " -> ".join(result["visits"]))

# Langfuse는 기록을 모아서 보내므로, flush()로 남은 기록을 바로 전송합니다.
langfuse.flush()

# 청구마다 Langfuse trace 주소입니다. 주소를 열면 그 실행의 trace 상세 화면이 나옵니다.
for number, (claim_id, run) in enumerate(claim_runs.items(), start=1):
    print(f"{number}. {claim_id}:", run["trace_url"])

**확인할 점:** 5-3 표대로라면 청구마다 claim_worker·terms_worker·reply_writer를 한 번씩 거치면 됩니다. 방문 순서가 이와 다른지 봅니다. 출력된 trace 주소를 열어 5-4의 화면을 직접 확인합니다.

### 5-4. Langfuse 화면 읽기

**화면 여는 법**

1. 5-3에서 출력한 trace 주소를 누릅니다. 주소 없이 열 때는 cloud.langfuse.com 에 로그인하고 위쪽에서 프로젝트를 고른 뒤 왼쪽 메뉴의 **Tracing**을 누릅니다.
2. 목록에서 행 하나를 누르면 그 trace의 상세 화면이 열립니다.
3. 상세 화면 위의 탭으로 보기를 바꿉니다. **Tree**는 실행 순서 트리, **Timeline**은 시간 막대, **Graph**는 노드 연결과 실행 횟수입니다.
4. 트리에서 항목 하나를 누르면 오른쪽에 그 항목의 정보가 나옵니다. 맨 위 줄은 시각·걸린 시간·비용·토큰, **Preview**는 입력(Input)과 출력(Output)입니다.

아래 화면마다 붙인 **주소**는 브라우저 주소창에 보이는 형식입니다. `cloud.langfuse.com`은 `.env`의 `LANGFUSE_BASE_URL` 호스트이고, `<프로젝트 ID>`는 프로젝트를 고른 뒤 주소창의 `/project/` 다음 값, `<trace id>`는 5-3에서 출력한 주소의 끝부분입니다. 트리에서 항목을 누르면 주소 끝에 `?observation=<항목 id>`가 붙습니다.

화면의 다른 메뉴와 요소는 [부록: Langfuse 화면 안내](부록_Langfuse_화면_안내.ipynb)에 모아 두었습니다.

**trace 목록**

- 주소: `https://cloud.langfuse.com/project/<프로젝트 ID>/traces`
- 누를 곳: 왼쪽 메뉴 **Tracing**, 오른쪽 위 **Columns**에서 Trace Tags·Session ID 켜기, 위쪽 검색바에 `name:=claim-team traceTags:v1` 입력. 노트북을 여러 번 실행했다면 앞선 실행의 trace도 함께 나오므로 시각이 가장 최근인 행을 봅니다.
- 보여 주는 것: 행 하나가 trace 하나(청구 한 건의 실행)입니다. Latency는 걸린 시간, Trace Tags는 태그, Session ID는 세션입니다. Session ID 값을 누르면 그 세션(그 청구)의 trace만 모은 화면이 열립니다.

<img src="images/langfuse/l1_trace_list.png" width="1050" alt="Langfuse Tracing 목록의 검색바에 기본 조건 isRootObservation:true와 함께 name:=claim-team traceTags:v1 조건이 들어간 화면에서 입력한 조건, claim-team trace 2개의 Trace Tags 열(day56, v1), Session ID 열(claim-review-2026-10-08-CLM-2102, claim-review-2026-10-08-CLM-2101)을 빨간 상자로 강조한 화면">

**확인할 점:** 청구 2건의 trace가 태그 `day56`, `v1`은 같고, 세션은 청구마다 `claim-review-2026-10-08-CLM-2101`처럼 thread_id와 같은 이름으로 따로 있습니다.

**trace 상세의 Tree**

- 주소: `https://cloud.langfuse.com/project/<프로젝트 ID>/traces/<trace id>` (5-3에서 출력한 CLM-2101 주소)
- 누를 곳: 목록에서 CLM-2101 행, 상세 화면의 **Tree** 탭
- 보여 주는 것: 실행 순서대로 쌓인 기록입니다. 들여쓰기가 깊을수록 안쪽 실행이고, 항목마다 걸린 시간과 비용이 붙습니다. 빨간 Error 표시는 예외로 끝난 항목입니다.

<img src="images/langfuse/l1_trace_tree.png" width="1050" alt="Langfuse trace 상세의 Tree 탭에서 claim-team 아래 supervisor 다음 claim_worker 노드 한 번의 실행(claim_agent, model 안의 ChatOpenAI, tools 안의 get_claim, 다시 model)을 빨간 상자로 강조한 화면">

**확인할 점:** 빨간 상자가 claim_worker 노드 한 번의 실행입니다. 워커 노드 아래에 에이전트(claim_agent)가, 그 아래에 모델 호출(ChatOpenAI)과 도구 호출(get_claim)이 있습니다. 4절 스트리밍의 `팀/claim_worker > model` 줄이 이 트리의 한 칸입니다. terms_worker가 실행된 trace라면 `search_terms` 아래에 검색기 호출이 붙어 검색된 조항을 볼 수 있습니다.

**Graph(Agent Graph)**

- 주소: `https://cloud.langfuse.com/project/<프로젝트 ID>/traces/<trace id>` (CLM-2101, 같은 화면에서 탭만 바꿈)
- 누를 곳: CLM-2101 trace 상세의 **Graph** 탭, 왼쪽 위 **Aggregated**
- 보여 주는 것: 노드 사이의 연결과, 같은 이름 노드를 합친 실행 횟수입니다. 노드 이름 옆 괄호 `(왼쪽/오른쪽)`의 **오른쪽 숫자가 이 trace에서 그 노드가 실행된 횟수**입니다. 왼쪽 숫자는 그 노드를 누를 때 오른쪽 패널에 열리는 실행 기록을 가리키는 표시입니다. 처음에는 오른쪽 숫자와 같고, 같은 노드를 다시 누를 때마다 하나씩 줄면서 다른 실행 기록으로 넘어갑니다(5/5, 4/5, … 1/5 다음은 다시 5/5). 그래서 처음 화면에서는 두 숫자가 같습니다. 한 번만 실행된 노드에는 괄호가 붙지 않습니다. LangGraph 실행이면 따로 설정하지 않아도 그려집니다. Expanded를 누르면 합치지 않고 실행 순서대로 펼칩니다.

<img src="images/langfuse/l1_agent_graph.png" width="760" alt="Langfuse Graph 탭의 Aggregated 보기에서 claim_worker (5/5)와 supervisor (10/10)을 빨간 상자로 강조한 화면. terms_worker는 (3/3), reply_writer는 횟수 없이 한 번 표시된다">

**확인할 점:** 횟수는 실행마다 다르므로 자기 trace에서 claim_worker 숫자가 1보다 큰지 봅니다. 청구 조회는 `get_claim` 한 번이면 끝나는 일이므로, 1보다 크면 같은 일을 반복한 실행입니다. Graph 탭의 아이콘은 Tree의 종류 표시와 다를 수 있으니, observation 종류는 Tree에서 확인합니다.

**모델 호출 상세**

- 주소: `https://cloud.langfuse.com/project/<프로젝트 ID>/traces/<trace id>?observation=<항목 id>` (CLM-2101에서 그 ChatOpenAI를 누른 주소)
- 누를 곳: CLM-2101 trace의 **Tree**에서 첫 supervisor 아래 **ChatOpenAI**
- 보여 주는 것: 맨 위 줄은 호출 시각·걸린 시간·비용·토큰 수·모델 이름입니다. Preview의 System·User는 모델이 받은 메시지, Assistant는 모델의 답입니다.

<img src="images/langfuse/l1_generation_detail.png" width="1050" alt="Langfuse에서 첫 supervisor 아래 ChatOpenAI 모델 호출을 연 화면으로, 트리의 그 기록, 2.10s·$0.000229·671 tokens·gpt-5.6-luna 줄, 심사 요청과 지금까지 보고(아직 없음)가 든 User 메시지를 빨간 상자로 강조한 화면">

**확인할 점:** User 메시지에 "심사 요청"과 "지금까지 보고"가 있습니다. 첫 판단이라 '지금까지 보고'는 아직 없음입니다. supervisor가 무엇을 보고 판단했는지는 이 입력이 정확히 알려 줍니다. Assistant 칸에는 'Thinking (redacted)'(암호화되어 내용이 보이지 않는 모델의 추론)만 보이고 Route 답은 비어 있을 수 있습니다. 고른 워커와 지시는 바로 위 **supervisor** 노드를 누르고 Output의 `goto`, `instruction`에서 봅니다.

## 6. trace로 실패 원인 찾기

### 6-1. 증상: 같은 워커 반복과 결론 없는 답변

5-3의 표대로라면 청구마다 청구 조회, 약관 검색, 답변 작성을 거쳐 지급 또는 부지급으로 결론이 나야 합니다. CLM-2102는 제9조에 따라 보상하지 않는다고 안내해야 합니다. LLM 실행이라 결과는 실행마다 다를 수 있으므로, 아래 셀의 출력으로 이번 실행을 판단합니다.

먼저 청구마다 워커별 방문 횟수와 처리 상한(`MAX_TURNS`)에 닿았는지 셉니다. 모델을 부르지 않고 5-3에서 모은 `claim_runs`만 읽습니다.

In [ ]:
from collections import Counter

# 청구마다 워커별 방문 횟수와, 처리 상한(MAX_TURNS)에 닿았는지 봅니다.
for claim_id, run in claim_runs.items():
    worker_counts = Counter(run["visits"])
    reached_limit = run["turn"] >= MAX_TURNS
    print(claim_id, "워커별 방문:", dict(worker_counts))
    print("    처리 상한 도달:", reached_limit)

**확인할 점:** claim_worker의 방문이 다른 워커보다 많고 처리 상한에 닿았다면 첫째 증상입니다. `get_claim`은 한 번에 청구 항목을 모두 돌려주므로 청구 조회를 여러 번 할 이유가 없습니다. 처리 상한(55일차)이 반복을 멈췄을 뿐, 반복한 이유는 아직 모릅니다.

다음은 CLM-2102의 고객 답변입니다.

In [ ]:
# CLM-2102의 마지막 보고가 reply_writer의 고객 답변입니다.
reply_2102 = claim_runs["CLM-2102"]["results"][-1]["result"]
display(Markdown(reply_2102))

In [ ]:
# 자기검산: 결론 없이 심사 담당자에게 넘겼는지, 파업 제외 조항(제9조)으로 결론을 냈는지 봅니다.
print("답변에 '심사 담당자 확인'이 있는가:", "심사 담당자 확인" in reply_2102)
print("답변에 제9조가 있는가:", "제9조" in reply_2102)

**확인할 점:** 답변이 지급 여부를 정하지 못하고 '심사 담당자 확인 필요'로 끝났다면 둘째 증상입니다. 처리 상한에 닿으면 supervisor가 reply_writer에게 이 안내를 맡기기 때문입니다. 청구액처럼 claim_worker가 조회했어야 할 값까지 '보고에 없다'고 쓰는 경우도 있습니다.

두 증상 모두 마지막 노드에서 보이지만, reply_writer는 받은 보고대로 썼고 supervisor도 받은 보고대로 같은 워커를 다시 골랐을 수 있습니다. 거꾸로 올라갑니다.

### 6-2. 거꾸로 따라가기

CLM-2102 trace를 열고 아래 순서로 확인합니다. 단계마다 묻는 것은 하나입니다. "청구액·증빙·지연 사유가 여기에 있는가?"

1. Agent Graph: 5-4에서 본 것처럼 claim_worker의 실행 횟수가 큰가
2. supervisor 입력: claim_worker를 다시 고른 supervisor 모델 호출의 '지금까지 보고'에 청구액·증빙·지연 사유가 있는가
3. claim_worker 출력: claim_worker가 돌려준 보고에 청구액·증빙·지연 사유가 있는가

값이 있던 곳과 사라진 곳 사이가 원인이 있는 자리입니다.

**supervisor 입력**

- 주소: `https://cloud.langfuse.com/project/<프로젝트 ID>/traces/<trace id>?observation=<항목 id>` (5-3에서 출력한 CLM-2102 주소에서 그 ChatOpenAI를 누른 주소)
- 누를 곳: CLM-2102 trace의 **Tree**에서 첫 claim_worker 다음 supervisor 아래 **ChatOpenAI**, Preview의 User
- 보여 주는 것: supervisor가 판단할 때 실제로 받은 문장입니다. 빨간 상자 안이 '지금까지 보고'의 claim_worker 줄입니다.

<img src="images/langfuse/l1_bug_supervisor_input.png" width="1050" alt="CLM-2102 trace에서 첫 claim_worker 다음 supervisor 모델 호출의 User 메시지로, 지금까지 보고 안의 claim_worker 보고가 청구인·사고 종류 줄 다음에서 끊겨 청구액·증빙·지연 사유가 없는 부분을 빨간 상자로 강조한 화면">

**확인할 점:** '지금까지 보고'의 claim_worker 줄이 문장 중간에서 끊겨 청구액·증빙·지연 사유가 없습니다. 바로 위 supervisor 노드의 Output(`instruction`)이나 다음 claim_worker 노드의 Input을 보면, supervisor가 청구액·증빙·지연 사유를 아직 확인하지 못했다고 보고 같은 조회를 다시 맡깁니다. 4절 스트리밍 출력의 '지시' 줄에도 비슷한 말이 나올 수 있습니다.

**claim_worker 출력**

- 주소: `https://cloud.langfuse.com/project/<프로젝트 ID>/traces/<trace id>?observation=<항목 id>` (같은 CLM-2102 trace에서 그 ChatOpenAI를 누른 주소)
- 누를 곳: 바로 앞 **claim_worker**를 펼쳐 claim_agent 아래 마지막 **ChatOpenAI**, Preview의 Assistant
- 보여 주는 것: 워커가 만든 보고 전체입니다. claim_worker 노드의 Output에도 같은 보고가 있지만 한 줄 문자열로 접혀 보이므로 모델 호출 화면에서 읽습니다. 빨간 상자 안이 supervisor 입력에서 사라진 지연 사유·청구 금액·증빙입니다.

<img src="images/langfuse/l1_bug_worker_output.png" width="1050" alt="CLM-2102 trace에서 첫 claim_agent의 마지막 모델 호출을 연 화면으로, 보고 전체의 지연 사유(출발 3일 전 공지된 항공사 노조 파업), 청구 항목, 청구액 150,000원, 제출 증빙을 빨간 상자로 강조한 화면">

In [ ]:
# CLM-2102 실행의 보고 목록입니다. State의 results에는 워커가 돌려준 보고가 잘리지 않고 그대로 들어 있습니다.
results_2102 = claim_runs["CLM-2102"]["results"]

# 첫 번째 claim_worker 보고를 꺼냅니다. claim_worker가 한 번도 불리지 않았으면 빈 문자열로 남습니다.
claim_report = ""
for row in results_2102:
    if row["worker"] == "claim_worker":
        claim_report = row["result"]
        break

print("claim_worker 보고 글자 수:", len(claim_report))
print("claim_worker 보고 전체:", claim_report)

같은 보고가 `format_results`를 거치면 어떤 문자열이 되는지 봅니다. supervisor 입력 화면의 claim_worker 줄과 같은 문자열입니다.

In [ ]:
# 같은 보고가 format_results를 거치면 supervisor 프롬프트에 어떤 문자열로 들어가는지 봅니다.
cut_text = format_results([{"worker": "claim_worker", "result": claim_report}])
print("format_results가 만든 문자열:", cut_text)

**확인할 점:** 3-2의 `format_results`가 토큰을 아끼려고 보고마다 앞 60자만 넣습니다. 보고는 청구번호·청구인·사고 종류 같은 앞 항목부터 쓰므로, 60자 안에 청구액·증빙·지연 사유까지 들어가지 못합니다. supervisor와 세 워커(`run_worker`)는 모두 이 함수를 거친 문자열만 받습니다.

| 단계 | 무엇을 봤나 | 청구액·증빙·지연 사유 | 그 결과 |
|---|---|---|---|
| claim_worker 출력 | 워커 보고 전체 | 있음 | 조회는 끝남 |
| `format_results` | 보고마다 앞 60자 | 잘림 | - |
| supervisor 입력 | '지금까지 보고' | 없음 | 같은 조회를 여러 번 다시 맡김, 약관 검색이 늦어지거나 빠짐 |
| reply_writer 입력 | '팀이 지금까지 확인한 내용' | 없음 | 처리 상한에 닿아 결론 없이 '심사 담당자 확인 필요' |

증상은 "이미 끝낸 단계 반복"(15.7%)이었고, 원인은 "앞선 대화 내용을 잃음"입니다. 전달 코드가 보고를 잘라 supervisor가 앞 보고를 받지 못했습니다. 둘 다 1-2 표의 "시스템 설계 문제"에 속합니다. 에이전트가 정보를 숨긴 것(에이전트 간 어긋남)이 아닙니다. 처리 상한은 반복을 멈춰 비용을 막았지만 원인을 고치지는 못합니다.

### 6-3. 고치고 비교하기

`format_results`를 같은 이름으로 다시 정의해 보고를 자르지 않게 합니다.

In [ ]:
# [보조 함수] format_results 수정: 같은 이름으로 다시 정의해, 보고를 자르지 않고 그대로 넣습니다.
def format_results(results):
    """워커 보고 목록을 자르지 않고 '[워커] 보고' 문단으로 묶습니다."""
    if not results:
        return "아직 없음"

    lines = []
    for row in results:
        lines.append(f"[{row['worker']}] {row['result']}")

    return "\n\n".join(lines)

supervisor와 `run_worker`는 실행할 때마다 `format_results`라는 이름으로 함수를 찾습니다. 같은 이름으로 다시 정의했으므로 그래프를 다시 컴파일하지 않고 같은 `team_app`으로 실행합니다. 태그를 `v2-full-report`로 바꿔 청구 2건을 다시 실행합니다. 수정 후 실행은 v1을 이어 하는 실행이 아니라 새 작업이므로 thread_id·세션 이름 끝에 `-v2`를 붙입니다.

**주의:** 위 셀 뒤로는 `team_app`이 수정한 함수로 실행됩니다. 5-3이나 6-2 셀을 다시 실행하면 수정 후 함수로 실행돼 v1 결과가 나오지 않습니다. v1 결과를 다시 보려면 커널을 다시 시작하고 처음부터 실행합니다.

In [ ]:
# 같은 team_app으로, v2-full-report 태그를 달아 청구 2건을 다시 실행합니다.
claim_runs_v2 = {}
for claim_id in CLAIMS:
    # 수정 후 실행은 v1을 이어 하는 실행이 아니라 새 작업이므로, thread_id·세션 이름 끝에 -v2를 붙여 나눕니다.
    run_id = f"claim-review-{TODAY}-{claim_id}-v2"
    config = make_config(run_id, run_id, ["day56", "v2-full-report"])
    result = team_app.invoke(claim_input(claim_id), config)

    handler = config["callbacks"][0]
    claim_runs_v2[claim_id] = {
        "visits": result["visits"],
        "turn": result["turn"],
        "results": result["results"],
        "trace_url": langfuse.get_trace_url(trace_id=handler.last_trace_id),
    }
    print(claim_id, "방문 순서:", " -> ".join(result["visits"]))

langfuse.flush()

**비교:** 청구마다 v1과 v2의 워커 방문 수와 처리 상한 도달 여부를 나란히 봅니다.

In [ ]:
# v1과 v2에서 청구마다 워커 방문 수와 처리 상한 도달 여부를 나란히 봅니다.
for claim_id in CLAIMS:
    v1_run = claim_runs[claim_id]
    v2_run = claim_runs_v2[claim_id]
    print(claim_id, "방문 수 v1:", len(v1_run["visits"]), "/ v2:", len(v2_run["visits"]))
    print("    처리 상한 도달 v1:", v1_run["turn"] >= MAX_TURNS, "/ v2:", v2_run["turn"] >= MAX_TURNS)

**확인할 점:** v2에서 방문 수가 줄고 처리 상한에 닿지 않는지 봅니다. 보고 전체가 전달되면 supervisor가 청구 조회를 다시 맡길 이유가 없어집니다. v2에서도 같은 워커를 몇 번 다시 부를 수 있습니다(예: 서류 확인을 위한 재조회, reply_writer 재호출). 처리 상한에 닿지 않고 결론을 냈는지가 기준입니다.

CLM-2102의 v2 답변이 '심사 담당자 확인' 없이 제9조로 결론을 냈는지 확인합니다.

In [ ]:
# 자기검산: 수정 후(v2) CLM-2102 답변이 제9조로 결론을 냈는지 봅니다.
reply_2102_v2 = claim_runs_v2["CLM-2102"]["results"][-1]["result"]
print("v2 답변에 '심사 담당자 확인'이 있는가:", "심사 담당자 확인" in reply_2102_v2)
print("v2 답변에 제9조가 있는가:", "제9조" in reply_2102_v2)
print("v2 trace:", claim_runs_v2["CLM-2102"]["trace_url"])

수정 후(v2) 실행에서 CLM-2102의 고객 답변입니다. 5-3의 표대로 제9조에 따라 보상하지 않는다고 안내하는지 봅니다. 실행마다 문장은 다를 수 있으므로, 위 셀의 두 검사와 함께 판단합니다.

In [ ]:
# 수정 후(v2) 실행에서 CLM-2102의 고객 답변입니다.
display(Markdown(reply_2102_v2))

**태그로 나란히 보기**

- 주소: `https://cloud.langfuse.com/project/<프로젝트 ID>/traces`
- 누를 곳: 왼쪽 메뉴 **Tracing**, 위쪽 검색바에 `name:=claim-team` 입력(5-4의 조건에서 `traceTags:v1`만 지움)
- 보여 주는 것: v1과 v2 trace가 한 목록에 나옵니다. 빨간 상자 안이 Latency(걸린 시간) 열과 Trace Tags 열입니다. 검색바에 `traceTags:v1`이나 `traceTags:v2-full-report`를 더하면 한쪽만 걸러 볼 수 있습니다.

<img src="images/langfuse/l1_compare_tags.png" width="1050" alt="Langfuse Tracing 목록의 검색바에 기본 조건 isRootObservation:true와 함께 name:=claim-team 조건만 들어간 화면에서 v2-full-report 태그 trace와 v1 태그 trace가 함께 나오고, Latency 열과 Trace Tags 열을 빨간 상자로 강조한 화면">

**확인할 점:** 태그로 고치기 전과 후의 trace를 구분해 열 수 있습니다. 실행 시간은 실행마다 달라지므로 v2 trace를 열어 워커 방문이 줄었는지, supervisor 입력의 '지금까지 보고'에 claim_worker 보고 전체가 들어 있는지로 판단합니다.

## 7. 운영에서 관측 설정

Anthropic은 자사 멀티에이전트 리서치 시스템이 일반 채팅보다 약 15배 많은 토큰을 쓴다고 밝혔습니다. 기록할 호출도 그만큼 많아지므로, 운영에서는 무엇을 얼마나 어떻게 남길지 정해 둡니다.

설정을 넣는 곳은 두 가지입니다. 실행마다 달라지는 값(세션·태그)은 실행할 때 넘기는 config의 `metadata`에 넣고, 프로그램 전체에 하나인 값(release·sample_rate·mask·environment)은 Langfuse 클라이언트를 만들 때 넣습니다. 클라이언트 값은 `.env`에 한 줄로 적어도 되며, 그러면 코드를 고치지 않고 바꿀 수 있습니다.

| 설정 | 하는 일 | 넣는 곳 | 예 |
|---|---|---|---|
| 세션 | 한 작업의 실행(첫 실행, 멈춘 단계부터 다시 실행한 것)을 한 세션으로 묶습니다. 체크포인터로 이어 하려면 `thread_id`에도 같은 값을 넣습니다 | 실행 config | `metadata`의 `langfuse_session_id`, `configurable`의 `thread_id` (5-2의 `make_config`가 넣음) |
| 태그 | 수정 전후(v1, v2)처럼 실행을 나눠 걸러 봅니다. 실행마다 여러 개 달 수 있습니다 | 실행 config | `metadata`의 `langfuse_tags=["day56", "v1"]` |
| release | 배포 버전을 모든 실행에 같은 값으로 붙입니다 | 클라이언트 또는 `.env` | `Langfuse(release="2026-10-08")`, `LANGFUSE_RELEASE=2026-10-08` |
| sample_rate | 남길 실행의 비율입니다. 기본값 1.0은 모두 남깁니다 | 클라이언트 또는 `.env` | `Langfuse(sample_rate=0.2)`, `LANGFUSE_SAMPLE_RATE=0.2` |
| mask | 전화번호 같은 개인정보를 서버로 보내기 전에 가립니다 | 클라이언트 | `Langfuse(mask=함수)` |
| environment | 개발·운영 기록을 나눕니다. 기본값은 `default`입니다 | 클라이언트 또는 `.env` | `Langfuse(environment="production")`, `LANGFUSE_TRACING_ENVIRONMENT=production` |

- **sample_rate:** 일정 주기로 빼는 것이 아니라 실행(trace)마다 남길지를 정합니다. 남기는 실행은 모델·도구 기록까지 통째로 보내고, 빼는 실행은 아무것도 보내지 않습니다. 0.2면 실행 100번 중 약 20번만 남습니다. 빠진 실행은 나중에 볼 수 없으므로 개발 중에는 1.0으로 둡니다.
- **environment와 태그:** 태그는 실행마다 여러 개 다는 꼬리표이고 검색바(`traceTags:v1`)로 거릅니다. environment는 실행마다 하나만 정하는 값으로, 그 실행의 모든 기록에 붙고 화면 위쪽의 환경 선택으로 거릅니다. 이름은 영문 소문자·숫자·`-`·`_`만 쓰고, 한번 생기면 화면에서 지우거나 바꿀 수 없습니다.

**mask 함수 예:** 이 교안의 청구 데이터에는 연락처가 있어서 `get_claim` 도구 결과가 그대로 trace에 남습니다. 보내기 전에 휴대전화 번호를 가리는 함수를 만듭니다.

In [ ]:
import re

# 휴대전화 번호 모양(010-1234-5678)을 찾는 정규 표현식입니다.
PHONE_PATTERN = re.compile(r"01[016789]-\d{3,4}-\d{4}")


# [보조 함수] 전송 전 가리기: Langfuse가 보낼 값에서 휴대전화 번호를 가립니다.
def mask_phone(*, data, **kwargs):
    """문자열 속 휴대전화 번호를 가리고, dict·list는 안쪽 값마다 같은 처리를 합니다."""
    # Langfuse는 mask 함수를 data 키워드 인자로 부릅니다. 나머지 인자는 쓰지 않습니다.
    if isinstance(data, str):
        return PHONE_PATTERN.sub("***-****-****", data)

    # 도구 입력·출력은 dict나 list로 올 수 있어 안쪽 값까지 내려가며 가립니다.
    if isinstance(data, dict):
        masked = {}
        for key, value in data.items():
            masked[key] = mask_phone(data=value)
        return masked
    if isinstance(data, list):
        masked_list = []
        for value in data:
            masked_list.append(mask_phone(data=value))
        return masked_list

    return data


# CLM-2101의 연락처와 메모 속 다른 번호, 두 번호가 모두 가려지는지 mask_phone을 직접 불러 봅니다. Langfuse로는 보내지 않습니다.
sample = {"청구": "CLM-2101", "연락처": CLAIMS["CLM-2101"]["연락처"], "메모": "급하면 010-5555-1234로 연락 주세요"}
print("가린 결과:", mask_phone(data=sample))

**주의:** mask는 클라이언트를 만들 때 정합니다. `get_client()`는 처음 만든 클라이언트를 계속 돌려주므로, `Langfuse(mask=mask_phone)`를 노트북 맨 앞(2-1의 `get_client()`보다 먼저)에서 만들어야 합니다. 이 노트북은 이미 2-1에서 클라이언트를 만들었으므로 여기서는 함수만 확인합니다. 콜백이 보내는 입력·출력도 이 클라이언트를 거치지만, 운영에 쓰기 전에 Langfuse 화면에서 번호가 가려졌는지 직접 확인합니다.

```python
from langfuse import Langfuse

# get_client()보다 먼저 만들면, 뒤의 get_client()와 CallbackHandler가 이 설정을 씁니다.
langfuse = Langfuse(mask=mask_phone, sample_rate=0.2, environment="production")
```

## 8. 핵심 정리

- 궤적은 어떤 노드·에이전트·도구가 어떤 순서로 무엇을 받고 무엇을 냈는지의 기록입니다. 슈퍼바이저 팀은 워커 고르기와 워커 안 도구 고르기가 겹쳐 있어 궤적도 두 겹입니다.
- `stream(stream_mode="updates", subgraphs=True)`는 `(네임스페이스, 이벤트)`를 내줍니다. 네임스페이스로 팀 그래프의 이벤트와 워커 안 에이전트의 이벤트를 구별하고, 방문 횟수로 반복을 찾습니다.
- Langfuse는 실행 설정의 `callbacks`에 `CallbackHandler()`를, `metadata`에 `langfuse_session_id`와 `langfuse_tags`를 넣어 trace에 세션·태그를 붙입니다. 세션은 `thread_id`가 아니라 `langfuse_session_id`로 정해지며, 이 교안은 작업 하나 = thread 하나 = 세션 하나로 두 값을 같게 둡니다. 실행 뒤 `last_trace_id`로 trace 주소를 얻고 `flush()`로 전송합니다.
- 증상(같은 워커 반복, 결론 없는 답변)이 보인 노드와 원인이 생긴 곳은 다른 경우가 많습니다. 증상 노드의 입력에서 출발해 "그 정보가 여기에 있는가"를 물으며 거꾸로 올라가, 있던 곳과 사라진 곳 사이를 고칩니다. 이 교안에서는 워커 보고를 60자로 자르던 보조 함수였습니다.
- 수정 후 실행은 다른 태그를 달아 Tracing 목록에서 수정 전 실행과 비교합니다. 운영에서는 세션·태그·release·sample_rate·mask·environment로 무엇을 어떻게 남길지 정합니다.